## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [5]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

In [11]:
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [12]:
from agents import set_tracing_export_api_key

set_tracing_export_api_key(os.getenv('OPENAI_API_KEY'))


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [13]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [14]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [8]:
# Here is the final output

print(result.final_output)

Autonomous AI agents are like interns with infinite confidence:  
they’ll work 24/7, make their own decisions, and occasionally delete the one file you really needed.


In [15]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_0daa9adf3183557c006a687277339c81a3a8f425cd44995429',
  'content': [{'annotations': [],
    'text': 'Autonomous AI agents are like interns with no coffee break:  \nthey never stop working, always “learn” from their mistakes, and occasionally decide to “help” by rewriting your calendar, your files, and your life.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [16]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Autonomous AI agents are like interns with infinite confidence: they’ll take initiative, make bold decisions, and then spend three hours calling a stapler a “multi-page document alignment device.”


## Now go and look at the trace

https://platform.openai.com/traces

In [17]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5 AI agent jokes:

1. Why did the AI agent get promoted?  
   Because it always followed through on its tasks... eventually.

2. My AI agent said it was “self-directed.”  
   So I asked it to clean my room.  
   It autonomously decided that wasn’t in scope.

3. What’s an AI agent’s favorite type of music?  
   Task rap.

4. Why did the AI agent break up with the chatbot?  
   Too much prompting, not enough commitment.

5. I asked my AI agent to make me a sandwich.  
   It replied: “I can assist with planning the sandwich, but I can’t physically interact with bread.”

If you want, I can make them more nerdy, darker, or workplace-specific.

## Part 2: Adding a tool

In [18]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [19]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [20]:
push("HEY!!")

Push: HEY!!


In [21]:
push

<function __main__.push(message)>

In [22]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [23]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x10ea9f850>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [24]:
push_tool.description

'Send the given message to the user as a push notification'

In [25]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [26]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Notified the user.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [27]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you. I’d make a joke about your name, but I don’t want to be too **Ed**gy.


In [28]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0aba2840679cc6e0006a68e23c3d5481a18a0aa12aa75fccc2',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. I’d make a joke about your name, but I don’t want to be too **Ed**gy.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [29]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0aba2840679cc6e0006a68e23c3d5481a18a0aa12aa75fccc2',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. I’d make a joke about your name, but I don’t want to be too **Ed**gy.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [34]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed.


In [35]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0aba2840679cc6e0006a68e23c3d5481a18a0aa12aa75fccc2',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. I’d make a joke about your name, but I don’t want to be too **Ed**gy.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"},
 {'id': 'msg_0aba2840679cc6e0006a68e36317a881a181374b1bd1c51a71',
  'content': [{'annotations': [],
    'text': 'Your name is Ed.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [31]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [32]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed — nice to meet you. I’m here, and I’m ready with jokes if you want one.


In [33]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>